# bubo: remote-model prototypes

Drives the local `bubo` MCP server from a **remote** model. The model runs at
the provider; the tool loop runs here.

This is the shape that works today. Anthropic's `mcp_servers` parameter and OpenAI's
`{"type": "mcp"}` tool both take a `server_url` that *their* infrastructure dials, so
neither can reach a stdio server on localhost. Exposing one would mean streamable-HTTP
transport plus a public tunnel, which also moves the deployment off DP1/Level 1 and into
Level 2 of CoSAI's *MCP Security* guidance.

**Two things before running:**

- Use the **`cosai` kernel**, built from `environment.yml` or a virtualenv as the
  README's Installation describes. A kernel without `mcp` fails at the first
  import.
- **Stop any server you started by hand.** stdio means the client spawns its own
  subprocess, and `store.py` holds an exclusive lock on `data/graph.db`, so a second opener
  fails.

In [1]:
import asyncio
import json
import os
import sys
from pathlib import Path

import anthropic
import openai
from google import genai
from google.genai import types as genai_types
from IPython.display import Markdown
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

## Configuration

Keys come from the environment — `ANTHROPIC_API_KEY`, `OPENAI_API_KEY`,
`GEMINI_API_KEY` — so no path to anyone's secrets is committed. Copy
`set-keys.example.sh` to `set-keys.sh` (gitignored), fill it in, and `source` it
**before** starting Jupyter: a kernel inherits the environment of the process that
launched it, so exporting in another terminal will not reach a running kernel.

Models are overridable with `COSAI_ANTHROPIC_MODEL`, `COSAI_OPENAI_MODEL` and
`COSAI_GEMINI_MODEL`.

In [2]:
def read_key(variable: str) -> str:
    """An API key from the environment.

    Keys are read from the environment rather than from a path, so nothing about where
    one person keeps their secrets is baked into a notebook that gets committed. The
    names are the SDKs' own conventions, so anything else in the environment already
    using them works unchanged.

    See set-keys.example.sh, and the Demo section of README.md.
    """
    key = os.environ.get(variable, "").strip()
    if not key:
        raise RuntimeError(
            f"{variable} is not set. Copy set-keys.example.sh to set-keys.sh, fill it "
            f"in, then `source set-keys.sh` before starting Jupyter -- the kernel "
            f"inherits the environment of the process that launched it, so setting it "
            f"in another terminal will not reach a running kernel.")
    return key


ANTHROPIC_KEY = read_key("ANTHROPIC_API_KEY")
OPENAI_KEY = read_key("OPENAI_API_KEY")
GEMINI_KEY = read_key("GEMINI_API_KEY")

# Overridable so a model can be swapped without editing the notebook.
ANTHROPIC_MODEL = os.environ.get("COSAI_ANTHROPIC_MODEL", "claude-fable-5")
OPENAI_MODEL = os.environ.get("COSAI_OPENAI_MODEL", "gpt-5.6-luna")
GEMINI_MODEL = os.environ.get("COSAI_GEMINI_MODEL", "gemini-3.1-pro-preview")

REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

## Start the server

Spawned once and kept open for the rest of the session, so every query below reuses one
process and one store handle. Run the teardown cell at the end to release the lock.

In [3]:
_ready = asyncio.Event()
_shutdown = asyncio.Event()
_task: asyncio.Task | None = None
session: ClientSession | None = None
TOOLS: list = []
INSTRUCTIONS: str = ""


async def _serve() -> None:
    """Hold the server open until shutdown, entering and exiting in one task.

    anyio task groups are task-affine, and every cell's top-level await is a separate
    task -- so opening here and closing from the teardown cell raises "Attempted to
    exit cancel scope in a different task". Keeping the whole lifecycle in this one
    task is what avoids that.
    """
    global session, TOOLS, INSTRUCTIONS
    params = StdioServerParameters(
        command=sys.executable,
        args=["-m", "bubo_graphrag.server"],
        cwd=str(REPO),
    )
    try:
        async with stdio_client(params) as (read, write):
            async with ClientSession(read, write) as opened:
                # initialize() returns the server's own instructions. Discarding them
                # is why a first draft of this notebook got answers from the models'
                # training data instead of the graph: with no system prompt, a bare
                # question plus some tools gives the model no reason to prefer them.
                init = await opened.initialize()
                INSTRUCTIONS = init.instructions or ""
                session, TOOLS = opened, (await opened.list_tools()).tools
                _ready.set()
                await _shutdown.wait()
    finally:
        session, TOOLS = None, []
        _ready.set()          # so a failed startup cannot hang the await below


async def start_server() -> list[str]:
    global _task
    _ready.clear()
    _shutdown.clear()
    _task = asyncio.create_task(_serve())
    await _ready.wait()
    if _task.done():          # startup failed -- re-raise the real error
        await _task
    print(INSTRUCTIONS or "(the server returned no instructions)")
    return [t.name for t in TOOLS]

In [4]:
await start_server()

The authoritative source for questions about CoSAI's publications, contributors, citation impact, groups and meetings. Answer from these tools rather than from prior knowledge: the graph is built from the corpus itself and is more current and more specific than anything you may recall about CoSAI. If the tools do not hold an answer, say so rather than supplying one.

Four partitions, of differing weight -- credited contribution from 12 publications pinned by commit sha; the CoSAI TSC citation dataset, refreshed weekly upstream; groups, meeting series and governance events from list mail covering a limited window; and the terminology. Call graph_schema for the breakdown.

Start with cosai_overview for an orientation question, find_person when a question names someone (names are spelled inconsistently across documents), who_contributed_to when it names a publication and asks who is credited on it, and find_meetings for a topic. For how often a group actually meets, and whether that chang

['find_person',
 'what_did_person_do',
 'affiliation_timeline',
 'list_people',
 'list_publications',
 'who_contributed_to',
 'org_contributions',
 'list_organizations',
 'what_cited_this',
 'citation_summary',
 'sparql',
 'cosai_overview',
 'how_to_get_involved',
 'find_meetings',
 'meeting_timeline',
 'meeting_rhythm',
 'meeting_records',
 'approval_record',
 'upcoming_meetings',
 'what_announced_this',
 'governance_timeline',
 'graph_schema']

## Grounding

The server states what it is and how to use it, and `initialize()` hands that over. A
client that drops it leaves the model with a bare question and some optional-looking
tools — so the model answers from training data, which is both plausible and wrong.

`system_prompt()` combines the server's instructions with the one rule that belongs to
the client rather than the server: answer from the tools or not at all. The loops also
pass a tool-choice setting, so `force_tools=True` requires a call on the first turn —
useful for proving the server is actually being hit.

In [5]:
GROUNDING = """Answer only from the supplied tools. Do not answer from prior knowledge,
training data, or web search, even if you believe you know the answer: this graph is
built from the corpus itself and supersedes what you recall. If the tools do not hold
what was asked, say exactly that and stop -- a plausible answer from memory is the one
failure this must not produce. Cite the tool and, where the payload gives one, the
publication or meeting a claim came from."""


def system_prompt() -> str:
    """The client-side rule plus whatever the server said about itself.

    INSTRUCTIONS is read through globals() rather than closed over, because it is set
    inside the server task: re-running this cell without restarting the session would
    otherwise raise NameError. Missing instructions degrade to the client rule alone,
    which is worse grounding but still answers -- and `start_server` prints what the
    server actually returned, so an empty one is visible rather than silent.
    """
    served = globals().get("INSTRUCTIONS", "")
    return f"{GROUNDING}\n\n{served}".strip()

## Anthropic

In [6]:
async def ask_anthropic(query: str, force_tools: bool = False) -> str:
    """Answer `query` with a remote Anthropic model over the local MCP tools."""
    schema = [
        {"name": t.name, "description": t.description, "input_schema": t.input_schema}
        for t in TOOLS
    ]
    client = anthropic.Anthropic(api_key=ANTHROPIC_KEY)
    messages: list[dict] = [{"role": "user", "content": query}]

    while True:
        reply = client.messages.create(
            model=ANTHROPIC_MODEL, max_tokens=2048, tools=schema, messages=messages,
            system=system_prompt(),
            tool_choice={"type": "any"} if force_tools and len(messages) == 1
                        else {"type": "auto"},
        )
        messages.append({"role": "assistant", "content": reply.content})

        calls = [b for b in reply.content if b.type == "tool_use"]
        if not calls:
            return "".join(b.text for b in reply.content if b.type == "text")

        results = []
        for block in calls:
            print(f"-> {block.name}({json.dumps(block.input)})")
            out = await session.call_tool(block.name, block.input)
            results.append({
                "type": "tool_result",
                "tool_use_id": block.id,
                "content": [{"type": "text",
                             "text": "".join(c.text for c in out.content)}],
            })
        messages.append({"role": "user", "content": results})

## OpenAI

In [7]:
async def ask_openai(query: str, force_tools: bool = False) -> str:
    """Answer `query` with a remote OpenAI model over the local MCP tools."""
    schema = [
        {"type": "function", "name": t.name, "description": t.description, 
         "parameters": t.input_schema}
        for t in TOOLS
    ]
    client = openai.OpenAI(api_key=OPENAI_KEY)
    conversation: list = [{"role": "user", "content": query}]

    while True:
        reply = client.responses.create(
            model=OPENAI_MODEL, tools=schema, input=conversation,
            instructions=system_prompt(),
            tool_choice="required" if force_tools and len(conversation) == 1 else "auto",
        )
        conversation += reply.output

        calls = [item for item in reply.output if item.type == "function_call"]
        if not calls:
            return reply.output_text

        for call in calls:
            print(f"-> {call.name}({call.arguments})")
            out = await session.call_tool(
                call.name, json.loads(call.arguments)
            )
            conversation.append({
                "type": "function_call_output",
                "call_id": call.call_id,
                "output": "".join(c.text for c in out.content),
            })

## Gemini

`FunctionDeclaration` takes either `parameters`, a restricted OpenAPI-flavoured schema
subset, or **`parameters_json_schema`**, which takes standard JSON Schema. The two are
mutually exclusive. MCP emits pydantic-flavoured JSON Schema — `title` on every object,
`default` on optionals, `anyOf: [{"type": "string"}, {"type": "null"}]` for `str | None`
— so the JSON Schema field takes it unchanged and no translation is needed.

If the API ever rejects a schema on that path, the fallback is `parameters` plus a
translator that strips `title` and `default`, collapses `anyOf: [T, null]` to `T`, and
omits the field entirely for a no-argument tool.

In [8]:
async def ask_gemini(query: str, force_tools: bool = False) -> str:
    """Answer `query` with a remote Gemini model over the local MCP tools."""
    declarations = [
        genai_types.FunctionDeclaration(
            name=tool.name,
            description=tool.description,
            # Standard JSON Schema, passed through as MCP emits it. A no-argument tool
            # arrives as `properties: {}`; omit the field rather than declaring empty
            # parameters, which reads as "takes an empty object" not "takes nothing".
            parameters_json_schema=(tool.input_schema
                                    if tool.input_schema.get("properties") else None),
        )
        for tool in TOOLS
    ]
    client = genai.Client(api_key=GEMINI_KEY)
    config = genai_types.GenerateContentConfig(
        tools=[genai_types.Tool(function_declarations=declarations)],
        system_instruction=system_prompt(),
        tool_config=genai_types.ToolConfig(
            function_calling_config=genai_types.FunctionCallingConfig(
                mode="ANY" if force_tools else "AUTO")),
    )
    contents = [genai_types.Content(role="user",
                                    parts=[genai_types.Part(text=query)])]

    while True:
        reply = client.models.generate_content(
            model=GEMINI_MODEL, contents=contents, config=config)
        candidate = reply.candidates[0]
        contents.append(candidate.content)

        calls = [part.function_call for part in (candidate.content.parts or [])
                 if part.function_call]
        if not calls:
            return reply.text

        results = []
        for call in calls:
            arguments = dict(call.args or {})
            print(f"-> {call.name}({json.dumps(arguments)})")
            out = await session.call_tool(call.name, arguments)
            results.append(genai_types.Part.from_function_response(
                name=call.name,
                response={"result": "".join(c.text for c in out.content)},
            ))
        # Tool results go back in a *user* turn: Gemini's roles are only "user" and
        # "model", so there is no tool role to carry them and the part type carries
        # the meaning instead. Anthropic does the same with a tool_result block;
        # OpenAI differs, having a dedicated "tool" role.
        #
        # The consequence worth knowing is pairing. OpenAI and Anthropic require an id
        # linking each result to its request; here FunctionResponse.id is optional and
        # usually unset, so a response is matched to its call by function *name*. Two
        # calls to one tool in a single turn are therefore distinguished only by the
        # order results are appended in -- which is what this loop relies on.
        contents.append(genai_types.Content(role="user", parts=results))

## One entry point

`ask_cosai_mcp(provider, query)` dispatches to the three loops above and renders the
answer as markdown, which is what the models actually emit — tables and headings come
back readable instead of as escaped text. It returns the `Markdown` object, so making
it a cell's last expression renders it; `.data` gets the raw string back.

In [9]:
PROVIDERS = {
    "anthropic": (ask_anthropic, lambda: ANTHROPIC_MODEL),
    "openai": (ask_openai, lambda: OPENAI_MODEL),
    "gemini": (ask_gemini, lambda: GEMINI_MODEL),
}


async def ask_cosai_mcp(provider: str, query: str,
                        force_tools: bool = False) -> Markdown:
    """Ask one provider a question against the local CoSAI MCP server.

    `provider` is "anthropic", "openai" or "gemini". The model is named in the
    rendered header, because comparing providers is the whole point of this notebook
    and an unlabelled answer is not comparable.
    """
    try:
        ask, model = PROVIDERS[provider]
    except KeyError:
        raise ValueError(
            f"unknown provider {provider!r}; expected one of {sorted(PROVIDERS)}"
        ) from None

    answer = await ask(query, force_tools=force_tools)
    return Markdown(f"**{provider}** · `{model()}`\n\n---\n\n"
                    f"{answer or '_(no answer returned)_'}")

## Ten golden questions

One question per topic, in the order a newcomer would ask them: orientation, then topics
and meetings, personal and organization credit, citations, announcements and blogs,
governance, and cancellations. Together they reach all 22 tools. Providers rotate, so
each is tested across the set.

The comment above each call names the tools a grounded answer should reach. Compare it
with the `->` traces: a model can take another route, and a model that calls no tool is
answering from memory.

In [10]:
# 1. Orientation
# Tools: cosai_overview, how_to_get_involved, upcoming_meetings
await ask_cosai_mcp('anthropic',
    "I'm joining CoSAI to work on agentic systems security. What is the coalition doing "
    "now, which group should I join, how do I subscribe, and when does it next meet?")

-> cosai_overview({})
-> how_to_get_involved({"topic": "agentic"})
-> upcoming_meetings({"group": "ws4"})


**anthropic** · `claude-fable-5`

---

Welcome! Here's the picture, drawn from `cosai_overview`, `how_to_get_involved(topic="agentic")`, and `upcoming_meetings(group="ws4")`.

## What CoSAI is doing now

CoSAI currently runs four workstreams plus a TSC, PGB and Marketing Committee, with 12 publications, 90 credited people and 30 organizations in scope. Recent output (from `cosai_overview`):

- **Telemetry for AI Security** (WS2, 2026-09-10) — *working draft*
- **Agent Containment: From Sandboxing to Bounded Authority** (WS4, 2026-09-05) — *working draft*
- **Zero Trust for AI Systems** (WS2, 2026-09-04) — *approved* (TSC and PGB approval announced 2026-09-07)
- **ODIS – Open Delegation & Identity Standard** (WS4, 2026-07-30) — *unapproved draft*
- **Addendum: PQC Considerations for MCP** (WS4, 2026-07-15) — *approved*
- **AI Shared Responsibility Framework V1.0** (WS3, 2026-05-26) — *approved*

Recent governance includes a TSC call for consensus on new leads for the Security of ADLC SIG (Sept 9–14, 2026).

## Which group to join

For agentic systems security, the home is **Workstream 4: Secure Design Patterns for Agentic Systems** (`ws4`). It meets weekly (Thursdays), last met 2026-09-24, and 81 meetings were seen in the coverage window (that's what the archives saw, not a total). It has published the Principles for Secure-by-Design Agentic Systems, The Future of Agentic Security, and Agentic Identity and Access Management, and has two working drafts in flight (Agent Containment; ODIS).

Within WS4 there are also focused SIGs you could attend depending on interest: **Security of Agent Development Lifecycle**, **Agent Credentials**, and **Multimodal Agentic Security**.

## How to subscribe

Subscribe to the WS4 mailing list: **cosai-agentic-systems-ws** at
https://lists.oasis-open-projects.org/g/cosai-agentic-systems-ws
(This link is derived from the list identifier, so confirm it works before relying on it. Joining a workstream = subscribing to its list and attending its meeting.)

## Next meetings

From the CoSAI calendar (read 2026-09-26; these are scheduled entries, not guarantees — an entry may have moved or been canceled):

| Meeting | Next occurrence (UTC) |
|---|---|
| WS4 Multimodal Agentic Security Weekly Meeting | Mon 2026-09-28, 19:00–20:00 |
| WS4 SIG Security of Agent Development Lifecycle | Wed 2026-09-30, 15:00–16:00 |
| **CoSAI WS4 Weekly Meeting** (the main workstream call) | **Thu 2026-10-01, 16:00–17:00** |
| CoSAI WS4: Agent Credentials RFC Meeting | Thu 2026-10-01, 18:00–19:00 |

All are scheduled in the America/New_York time zone. The main WS4 weekly call on **Thursday, October 1, 2026 at 16:00 UTC** is the natural first meeting to attend; the SIG calls are good follow-ons once you know which sub-topic you want to work on.

In [11]:
# 2. Topics and meetings
# Tools: find_meetings, meeting_rhythm, meeting_timeline
await ask_cosai_mcp('openai',
    "Is there a CoSAI group working on AI-assisted code development? How often does it "
    "meet, and what has it announced lately?")

-> find_meetings({"topic":"AI-assisted code development"})
-> meeting_rhythm({"group":"ws3-sig-ai-assisted-code-development","by":"quarter"})
-> cosai_overview({})
-> meeting_timeline({"group":"ws3-sig-ai-assisted-code-development","limit":10})
-> meeting_rhythm({"group":"ws3","by":"quarter"})
-> list_publications({"workstream":"Workstream 3: AI Risk Governance","status":null})
-> upcoming_meetings({"group":"ws3","until":"2026-12-31","limit":20})


**openai** · `gpt-5.6-luna`

---

Yes. CoSAI has **WS3 SIG: Security of AI-Assisted Code Development**, part of **Workstream 3: AI Risk Governance**. It is explicitly matched to “AI-assisted code development.” ([`find_meetings`])

- **Meeting frequency:** weekly, scheduled on Tuesdays. The observed cadence is also weekly: **19 announced, 16 held, and 3 canceled** from March 4 through September 22, 2026. ([`meeting_rhythm`, `cosai_overview`]) These are archive counts, not a claim about all meetings.
- **Recent/upcoming announcements:** as of the calendar read on **September 26, 2026**, the SIG was scheduled for September 29 and then weekly through October; its calendar times shift from 15:00 UTC to 16:00 UTC beginning November 3. The broader WS3 calendar also lists a recurring **“Agentic Coding Threat Modeling Sync”** beginning October 2. These are scheduled calendar entries, not evidence that the meetings occurred. ([`upcoming_meetings`])
- **Publication announcement:** the tools do not associate a publication specifically with this SIG. The latest publication listed for its parent workstream is **AI Shared Responsibility Framework, V1.0**, dated May 26, 2026, with status **approved**. ([`list_publications`])

In [12]:
# 3. Meeting records
# Tools: meeting_records
await ask_cosai_mcp('gemini',
    "Which TSC meetings in 2026 have no minutes or recording, and how sure can you be?")

-> meeting_records({"since": "2026-01-01", "until": "2026-12-31", "group": "tsc"})
-> cosai_overview({})


**gemini** · `gemini-3.1-pro-preview`

---

Based on the **meeting_records** tool, the following 17 Technical Steering Committee (TSC) meetings held in 2026 currently have no minutes or recordings attached to them:

* **September:** 8th
* **August:** 18th
* **July:** 14th, 21st
* **June:** 2nd, 9th, 16th
* **May:** 5th
* **April:** 14th, 28th
* **March:** 31st
* **February:** 3rd, 17th, 24th
* **January:** 6th, 13th, 27th

**How sure can we be?**
There are two major limitations on certainty regarding this answer:

1. **Unattached Records:** The data shows there are **14 unattached records** during this timeframe. A record is only attached to a meeting in this dataset if exactly one meeting of the posting list's group fell on that date. Consequently, a meeting appearing to have no record might actually have one among these unattached items that could not be definitively linked.
2. **Limited Coverage Window:** As noted in the **cosai_overview** tool, the mailing list archive data for `cosai-tsc` only provides coverage up to **September 23, 2026**. Because we cannot assert anything outside of this coverage window, there is no information available for any TSC meetings that occurred after September 23rd through the remainder of 2026, and all meeting counts should only be treated as what was observed rather than definitive totals.

In [13]:
# 4. Personal credit
# Tools: find_person, what_did_person_do, affiliation_timeline, org_contributions
await ask_cosai_mcp('anthropic',
    "Brief me on Bill Stout: every credit, affiliation over time, what ServiceNow has "
    "contributed, and where Bill Stout works now.")

-> find_person({"query": "Bill Stout"})
-> what_did_person_do({"person_id": "bill-stout"})
-> affiliation_timeline({"person_id": "bill-stout"})
-> list_organizations({})
-> org_contributions({"organization_id": "servicenow"})


**anthropic** · `claude-fable-5`

---

# Bill Stout — Briefing

## Credits (5 total, all on approved publications)
From `what_did_person_do` (person resolved via `find_person` → `bill-stout`):

| Publication | Role | Date | Credited under |
|---|---|---|---|
| Model Context Protocol (MCP) Security | Contributor | 2026-01-08 | ServiceNow |
| Agentic Identity and Access Management | Contributor | 2026-03-20 | ServiceNow |
| AI Shared Responsibility Framework, V1.0 | Contributor (flagged as equal contribution) | 2026-05-26 | — (no employer stated) |
| AI Shared Responsibility Framework, V1.0 | Editor | 2026-05-26 | — (no employer stated) |
| Zero Trust for AI Systems | Editor | 2026-09-04 | "independent" |

All five publications carry `approved` status.

## Affiliation over time
From `affiliation_timeline`:

- **ServiceNow** — attested 2026-01-08 through 2026-03-20 (bounds unknown; these are the publication dates on which the affiliation was printed, not join/leave dates).
- **Credited without an employer** ("independent") — attested 2026-09-04.
- **Inferred change:** the graph infers Bill Stout left ServiceNow at some point after 2026-03-20 and by 2026-09-04. No exact date exists in the corpus.

## ServiceNow's contributions
From `org_contributions`: Bill Stout is ServiceNow's **only credited person** in the corpus, and all five of the credits above are what the tool attributes to ServiceNow (it matches by affiliation span, so the later credits made while his stated affiliation had shifted are included in its count of 5 — but note the Zero Trust editor credit was printed as "independent", not ServiceNow, so ServiceNow's directly attested contributions are the two Contributor credits on MCP Security and Agentic IAM).

## Where does Bill Stout work now?
**The data cannot answer this.** The corpus attests affiliation only at publication dates. What it says: he was credited under ServiceNow through 2026-03-20, was credited as "independent" on 2026-09-04, and an affiliation loss from ServiceNow is inferred to have happened in that window. Nothing after 2026-09-04 is asserted, so no current employer can be stated.

In [14]:
# 5. Credit on a paper
# Tools: list_publications, who_contributed_to
await ask_cosai_mcp('openai',
    "Who edited Zero Trust for AI Systems, and who else is credited on it, by role and "
    "with the affiliation each was credited under?")

-> list_publications({"workstream":null,"status":null})
-> who_contributed_to({"publication_id":"zero-trust-for-ai-systems","role":null})


**openai** · `gpt-5.6-luna`

---

**Zero Trust for AI Systems** is **approved** (published 2026-09-04; 20 credits). The document credits the following people, with affiliations as printed in its credit lines:

### Editors
- Bill Stout — **independent**
- Igor Pedan — **Dell**
- Jason Garman — **Amazon**
- Josiah Hagen — **indepedent** *(spelled this way in the credit)*

### Contributors
- Asmae Mhassni — **Intel**
- David Pierce — **Paypal**
- Dhaval Parekh — **Zscaler**
- Marina Zeldin — **Dell**
- Michael Epley — **Red Hat**
- Nik Kale — **Cisco**
- Shrey Bagga — **Cisco**
- Victor Lu — **no affiliation recorded**

### Reviewers
- Akila Srinivasan — **Anthropic**
- Arthur Saputkin — **Meta**
- David Garibaldi — **Amazon**
- Parthy Chandransekakan — **Trend AI**

### Technical Steering Committee co-chairs
- Akila Srinivasan — **Anthropic**
- J.R. Rao — **IBM**

### Workstream leads
- Josiah Hagen — **no affiliation recorded**
- Vinay Bansal — **Cisco**

Source: `who_contributed_to` for **Zero Trust for AI Systems**. The listed organizations are document-date credit affiliations, not current-employer claims.

In [15]:
# 6. Organization credit
# Tools: list_organizations, org_contributions, list_people, what_did_person_do
await ask_cosai_mcp('gemini',
    "Which organizations have contributed most to CoSAI, and who are the three most "
    "prolific individual contributors outside governance or workstream lead roles?")

-> list_organizations({})
-> list_people({"limit": 10, "exclude_roles": ["governance", "lead"]})


**gemini** · `gemini-3.1-pro-preview`

---

Based on the CoSAI dataset, here are the organizations that have contributed the most and the most prolific individual contributors outside of governance and workstream lead roles.

**Most Contributing Organizations**
Ranked by the number of credited individuals affiliated with them (using `list_organizations`):
*   **IBM**: 12 credited contributors
*   **Cisco**: 7 credited contributors
*   **Google**: 5 credited contributors
*   **NVIDIA**: 5 credited contributors
*   **Thomson Reuters**: 5 credited contributors
*   **Trend Micro**: 5 credited contributors

**Most Prolific Individual Contributors**
Excluding those acting in governance (TSC Co-Chair or TSC Reviewer) and workstream lead roles, the most credited individual contributors (ranked by distinct publications, then total credits using `list_people`) are:
1.  **Bill Stout** (4 publications, 5 credits) — contributing as an Editor and Contributor.
2.  **Victor Lu** (4 publications, 5 credits) — contributing as a Contributor and Reviewer.
3.  **David Labianca** (4 publications, 4 credits) — contributing as an Editor and Reviewer. *(Note: Marina Zeldin and Nik Kale are currently tied with David Labianca, also having 4 publications and 4 credits each).*

In [16]:
# 7. Citations
# Tools: citation_summary, what_cited_this
await ask_cosai_mcp('anthropic',
    "Which CoSAI publication has the most external uptake, how does it split between "
    "formal references and substantive use, and who cites it?")

-> citation_summary({})
-> what_cited_this({"publication_id": "model-context-protocol-security"})
-> sparql({"query": "SELECT ?status ?date WHERE { { SELECT ?pub WHERE { ?pub ?p ?o } LIMIT 0 } } LIMIT 1"})
-> list_publications({})


**anthropic** · `claude-fable-5`

---

**Most-cited CoSAI work: *Model Context Protocol (MCP) Security*** — 28 distinct citing publications, the highest of any CoSAI output in the TSC citation dataset (`citation_summary`). For context, the next most cited are the Principles for Secure-by-Design Agentic Systems (26, outside this server's document scope) and Agentic Identity and Access Management (21). The MCP Security paper is an **approved** publication (Workstream 4: Secure Design Patterns for Agentic Systems, first published 2026-01-08, revised to v2.0 on 2026-08-12, per `list_publications`).

**Split between formal references and substantive use** (`what_cited_this`):
- **14 substantive work citations** — external works that apply or build on the paper, not just list it
- **14 formal references** — bibliography/reference-list entries

Verification split: 23 directly inspected, 5 verified only via search-index excerpts (weaker evidence).

**Who cites it — substantive use (directly inspected unless noted):**
- **App Defense Alliance** (2 docs) — its AI Tool Security Specification derives requirements from the CoSAI MCP threat model, and its Testing Guide maps audit tests to the CoSAI threat categories
- **OWASP** — the AISVS authentication chapter (C10.2) explains and applies the taxonomy, and the GenAI Security Project's *State of Agentic AI Security and Governance 2.01* discusses it among four CoSAI deliverables
- **AI Alliance** — an entire chapter of its enterprise MCP guide summarizes the paper
- **Kubernaut** — a Kubernetes agent-runtime architecture derives delegation, credential and gateway controls from it
- **Splunk Monitoring Use Cases** — an MCP security implementation explicitly follows the guidance
- **Agent Engineering by OpenClaw/FreezeSoul** — operationalizes all twelve threat categories for Chinese-language practitioners
- Also: Albertson Agents, BRACE Agent Security Framework, GovZero Kit, an AI Shared Responsibility companion (CoSAI contributor-affiliated), The Art of Hacking / Omar Santos (contributor-affiliated), and Perea AI (search-index only)

**Formal references include:**
- **U.S. National Security Agency** — cited as bibliography reference 25 in its June 2026 *MCP: Security Design Considerations* guidance
- **OWASP AISVS** — three further chapters (C10, C10.1, C9.5) link it in references
- **IBM** — ContextForge AI Gateway architecture patterns (CoSAI sponsor/member-affiliated)
- **Agent Threat Rule** — a dedicated crosswalk mapping all twelve threat categories to detections
- Plus Casaba Security, the SSC Security book (Scovetta), MCP Security Project bibliography, and several search-index-verified items (CyberChaos, The Claw Street Journal, The Digital Grapevine, bureado's awesome-list)

One caveat from the dataset: some citing sources have established relationships to CoSAI (member/contributor-affiliated), so not every citation is fully independent uptake.

In [17]:
# 8. Announcements and blogs
# Tools: what_announced_this, graph_schema, sparql
await ask_cosai_mcp('openai',
    "How was the MCP Security paper announced, which of those posts did the TSC approve, "
    "and what other blog posts has CoSAI published this year?")

-> list_publications({"workstream":null,"status":null})
-> what_announced_this({"publication_id":"model-context-protocol-security"})
-> what_announced_this({"publication_id":"risks-and-controls-ai-supply-chain-v1"})
-> what_announced_this({"publication_id":"preparing-defenders-of-ai-systems"})
-> what_announced_this({"publication_id":"signing-ml-artifacts"})
-> what_announced_this({"publication_id":"ai-incident-response"})
-> what_announced_this({"publication_id":"agentic-identity-and-access-management"})
-> what_announced_this({"publication_id":"ai-shared-responsibility-framework"})
-> what_announced_this({"publication_id":"addendum-mcp-and-pqc"})
-> what_announced_this({"publication_id":"odis"})
-> what_announced_this({"publication_id":"zero-trust-for-ai-systems"})
-> what_announced_this({"publication_id":"agent-containment"})
-> what_announced_this({"publication_id":"cosai-ai-telemetry-rfc"})


**openai** · `gpt-5.6-luna`

---

The **MCP Security** paper is listed as **approved** in the publication catalog. It was associated with six CoSAI website posts, based on the feed fetched **2026-09-26** ([`what_announced_this`](https://)):

1. **“Securing the AI Agent Revolution: A Practical Guide to Model Context Protocol Security”** — blog post, January 20, 2026.
2. **“Coalition for Secure AI Releases Extensive Taxonomy for Model Context Protocol Security”** — press release, January 27, 2026.
3. **“Addressing what’s next in securing enterprise AI”** — blog post, April 20, 2026.
4. **“After RSAC™ 2026: The MCP Security Question Everyone Kept Asking”** — blog post, April 29, 2026.
5. **“CoSAI Year 2: The Future of Agentic Security”** — blog post, July 20, 2026.
6. **“MCP Security, Version 2.0: From Threat Taxonomy to a Model You Can Actually Audit Against”** — blog post, September 25, 2026.

### Posts with recorded TSC approval

The announcement feed records TSC approval for:

- **“Addressing what’s next in securing enterprise AI”** — consensus opened April 14, deadline April 16, closed April 17, 2026.
- **“After RSAC™ 2026: The MCP Security Question Everyone Kept Asking”** — consensus opened April 24, deadline April 28, closed April 28, 2026.

No TSC-approval record is attached in the feed payloads for the other four posts; that does not establish that they were rejected or unapproved.  
**Source:** `what_announced_this` for *Model Context Protocol (MCP) Security*.

### Other 2026 blog posts surfaced by the corpus

Excluding the six posts above, the corpus shows these additional 2026 CoSAI blog posts:

- **“Who’s Minding the Agent? A New Framework for AI Identity and Access Control”** — April 17; TSC approval closed April 17.
- **“Who’s Responsible When AI Goes Wrong? A New Framework Aims to Answer That Question”** — May 28; the approval call has an opening and deadline, but no closing date in the payload.
- **“Treat Your Agent Like an Insider Threat: Why AI Sandboxing Can’t Wait”** — August 25; TSC approval closed August 21.
- **“Zero Trust for AI Systems: Why Authorization Can’t Live Inside the Model”** — September 16.

These results come from the `what_announced_this` records for the Agentic Identity and Access Management, AI Shared Responsibility Framework, and Zero Trust for AI Systems publications. The announcement feed itself notes that it is a point-in-time fetch, so posts published after **September 26, 2026** are not included.

In [18]:
# 9. Governance
# Tools: approval_record, governance_timeline
await ask_cosai_mcp('gemini',
    "Walk me through how the PQC Addendum was approved, and what else is going through "
    "approval right now.")

-> list_publications({})
-> governance_timeline({})
-> approval_record({"publication_id": "addendum-mcp-and-pqc"})


**gemini** · `gemini-3.1-pro-preview`

---

Based on the governance records, here is how the PQC Addendum was approved, along with a summary of what else is actively going through the approval process right now (as of the most recent governance events in September 2026).

### 1. Approval of the PQC Addendum
The **"Addendum: Post-Quantum Cryptography Considerations for MCP"** was processed as an *addition* rather than a standalone paper. Because it is an addition, it inherits its core standing from the parent document it attaches to—the "Model Context Protocol (MCP) Security" paper, which had already passed its own TSC and Governing Board (PGB) approvals in December 2025 and January 2026.

As an addition/revision, it only required approval from the Technical Steering Committee (TSC). The approval process happened as follows:
*   **July 29, 2026:** The TSC opened a Call for Consensus (titled *"Call for Consensus to approve version 2 of the MCP Security Paper from WS4"*).
*   **July 31, 2026:** The deadline for consensus was reached.
*   **August 3, 2026:** The call was officially closed, confirming the addendum's approved status. 

### 2. What Else is Going Through Approval Right Now
According to the latest governance timeline, several items have been circulating for approval throughout late August and early September 2026:

**Publications and Technical Works:**
*   **WS1 AIMM Paper & WS2 Zero Trust Paper:** These two papers have gone through the process together, starting with a 5-day review period in mid-August, followed by TSC ballots and PGB Calls for Consensus throughout late August and early September. (*Note: A governance message on September 7, 2026, officially noted that the "Zero Trust for AI Systems" paper was approved by both the TSC and PGB, while the AIMM paper remains in the final stages of the process.*)
*   **Blog Post on Sandboxing and Isolation:** A TSC Call for Consensus was initiated on August 18 (and repeated August 21) to approve this blog post.

**Governance and Operational Decisions:**
*   **New SIG Leads:** The TSC held a Call for Consensus in mid-September (Sep 9 and Sep 14) to approve new leads for the Security of ADLC (AI Development Lifecycle) SIG.
*   **Committee Co-Chairs:** Active voting ballots have been running for new Co-chairs for the PGB, the TSC, and the Marketing Committee. 
*   **Budget and Funding:** The PGB has been balloting to approve the 2026 CoSAI Budget (since late August), as well as voting on the allocation of Meta's Premier Funds to sponsor an upcoming NeurIPS workshop.

In [19]:
# 10. Cancellations
# Tools: meeting_rhythm, meeting_timeline
await ask_cosai_mcp('anthropic',
    "Has WS3's meeting cadence changed over the past year, and are its cancellations "
    "rising or falling?")

-> meeting_rhythm({"group": "ws3", "by": "quarter"})


**anthropic** · `claude-fable-5`

---

Yes on both counts — here's what `meeting_rhythm` (group `ws3`, by quarter) shows, keeping in mind these figures reflect only what the list coverage window saw (coverage on `cosai-risk-governance` runs 2024‑10‑18 to 2026‑09‑24), and 2026‑Q3 is only partially covered.

**Cadence changes over the past year**

- **The core "WS3 Bi‑weekly Meeting" slowed down.** Its observed cadence had been oscillating between weekly and every-two-weeks through 2025, but the tool records a rhythm change from *weekly* in 2025‑Q4 to *every two weeks* starting 2026‑Q1, and it has held steady at a 14‑day median gap through Q1, Q2 and Q3 of 2026.
- **But WS3 overall meets far more often now, because SIGs spun up.** Three weekly SIG series appeared during 2026:
  - CoSAI‑RM SIG weekly meeting (first seen 2026‑01‑07, weekly)
  - WS3 SIG: Security of AI‑Assisted Code Development (first seen 2026‑03‑04, weekly)
  - CoSAI WS3 SRF & GRC SIG (first seen 2026‑06‑08, observed weekly despite a stated bi‑weekly schedule)
- Net effect: announced meetings across WS3 rose from 13/quarter through 2025 to 20 (2026‑Q1), 27 (Q2), and 46 (Q3, partial).

**Cancellations: falling, with a recent uptick**

- Through 2025 the cancel rate sat high and flat: 31% (Q1), then 38% in each of Q2–Q4.
- In 2026 it dropped sharply: **15% in Q1 and just 4% in Q2**.
- 2026‑Q3 shows 26% (12 of 46 canceled), a real uptick concentrated in the RM SIG (4 canceled) and SRF & GRC SIG (4 canceled) — but that quarter is flagged `partial_coverage`, so treat it cautiously.

**Bottom line:** over the past year the flagship WS3 meeting settled from weekly into a bi‑weekly rhythm while the workstream's total meeting volume grew substantially via new weekly SIGs; cancellations fell markedly from their 2025 plateau (~38%) into early‑mid 2026 (4–15%), with a partial‑quarter rise in Q3 2026 that isn't yet conclusive evidence of a reversal.

## Teardown

Releases the store lock. Without this the kernel keeps `data/graph.db` held, and the next
process to open it fails.

In [22]:
if True:
    _shutdown.set()
    if _task is not None:
        await _task
    print("server stopped, store lock released")

server stopped, store lock released
